# 14 — GÉNÉRATION v2 : le groove 🥁🔥

La v1 (07) générait des frappes justes mais sur une grille **mécanique**. Ici on la fait **groover** en ajoutant ce qui rend le jembé vivant :
1. **Modèle d'ordre 2** : `P(frappe | position, 2 frappes précédentes)` avec **repli** (backoff) vers l'ordre 1 puis 0 quand le contexte est rare → des phrases plus cohérentes.
2. **Timing non-isochrone** : les subdivisions sont placées selon le **swing mesuré** (08), pas sur une grille régulière.
3. **Accelerando** : le tempo monte au fil des cycles, comme dans le vrai jeu.

Sonification par vraies frappes du corpus. Change `PIECE` / `SEED`.

## 0. Imports & données

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, soundfile as sf
from IPython.display import Audio, display
import djembe_utils as dj
strokes = pd.read_csv('output/all_jembe_strokes_typed.csv')
score   = pd.read_csv('output/partitions.csv')
c = dj.list_corpus(); VOC = ['-','b','t','s']

PIECE = 'Woloso'   # <- 'Manjanin', 'Maraka', 'Woloso'
SEED  = 7
takes = c.loc[c['piece']==PIECE, 'take'].unique()
rng = np.random.RandomState(SEED)

## 1. Apprendre le modèle d'ordre 2 (avec repli)

In [ ]:
sub = score[(score['piece']==PIECE) & (score['J1']!='?')].reset_index(drop=True)
c2, c1, c0 = {}, {}, {v: 1.0 for v in VOC}
p2 = p1 = '-'
for _, r in sub.iterrows():
    c2.setdefault((r['SD'], p2, p1), {v: 0.3 for v in VOC}); c2[(r['SD'], p2, p1)][r['J1']] += 1
    c1.setdefault((r['SD'], p1), {v: 0.3 for v in VOC});     c1[(r['SD'], p1)][r['J1']] += 1
    c0[r['J1']] += 1; p2, p1 = p1, r['J1']

def sample(sd, p2, p1):
    for tab, key in [(c2, (sd, p2, p1)), (c1, (sd, p1))]:
        d = tab.get(key)
        if d and sum(d.values()) > 2.5:                 # assez d'évidence -> on garde l'ordre élevé
            w = np.array([d[v] for v in VOC], float); return rng.choice(VOC, p=w/w.sum())
    w = np.array([c0[v] for v in VOC], float); return rng.choice(VOC, p=w/w.sum())
print(f'Modèle {PIECE} : {len(c2)} contextes d\'ordre 2, {len(c1)} d\'ordre 1')

## 2. Timing : swing mesuré + accelerando

In [ ]:
cp = pd.concat([dj.cycle_positions(PIECE, int(t)) for t in takes], ignore_index=True)
relpos = cp.groupby('SD')['rel'].mean().reindex(range(1,13)).values; relpos[0] = 0.0

NCYC, d0, d1 = 10, 2.1, 1.4     # nb cycles, durée cycle début -> fin (accelerando)
cycdur = np.linspace(d0, d1, NCYC)
starts = np.concatenate([[0], np.cumsum(cycdur)[:-1]])
iso = np.arange(12)/12
print('subdivision :', list(range(1,13)))
print('swing (pos) :', np.round(relpos,3))
print('grille méca :', np.round(iso,3), '  <- comparaison')
print(f'tempo : {60/(d0/4):.0f} -> {60/(d1/4):.0f} bpm')

## 3. Générer & sonifier

In [ ]:
def prof(code):
    return np.mean([p.values for t in takes if (p := dj.subdivision_profile(PIECE, int(t), code)) is not None], axis=0)
dprof, j2prof = prof('D1'), prof('J2')
def bankj1(s, n=8):
    q = strokes[(strokes['stroke']==s) & (strokes['code']=='J1') & (strokes['piece']==PIECE)].nlargest(300,'rms').sample(n, random_state=1)
    out = []
    for _, r in q.iterrows():
        y = dj.load_track(r['piece'], int(r['take']), r['code']); a = int(r['time']*dj.SR)
        cl = y[a:a+int(0.22*dj.SR)].copy(); cl[-400:] *= np.linspace(1,0,400); out.append(cl)
    return out
BJ1 = {s: bankj1(s) for s in ['b','t','s']}
yD = dj.load_track(PIECE,1,'D1'); onD = dj.load_onsets(PIECE,1,'D1')['Time'].values
BD = [yD[int(t*dj.SR):int(t*dj.SR)+int(0.25*dj.SR)] for t in onD[20:28]]
yJ2 = dj.load_track(PIECE,1,'J2'); onJ2 = dj.load_onsets(PIECE,1,'J2')['Time'].values
BJ2 = [yJ2[int(t*dj.SR):int(t*dj.SR)+int(0.20*dj.SR)] for t in onJ2[20:28]]

out = np.zeros(int((starts[-1]+cycdur[-1]+1)*dj.SR)); seq = []; p2 = p1 = '-'
for k in range(NCYC):
    for sd in range(1, 13):
        s = sample(sd, p2, p1); seq.append(s); p2, p1 = p1, s
        t0 = int((starts[k] + relpos[sd-1]*cycdur[k]) * dj.SR)
        if rng.rand() < dprof[sd-1]:  cl = BD[rng.randint(len(BD))];  out[t0:t0+len(cl)] += cl*0.9
        if rng.rand() < j2prof[sd-1]: cl = BJ2[rng.randint(len(BJ2))]; out[t0:t0+len(cl)] += cl*0.7
        if s != '-':                  cl = BJ1[s][rng.randint(len(BJ1[s]))]; out[t0:t0+len(cl)] += cl
out /= (np.abs(out).max() + 1e-9)
sf.write(f'output/generation_groove_{PIECE}.wav', out, dj.SR)
print(f'>>> {PIECE} groove ({len(out)/dj.SR:.1f} s) :'); display(Audio(out, rate=dj.SR))

## 4. Comparer v1 (grille méca) vs v2 (groove)

In [ ]:
import os
if os.path.exists(f'output/generation_{PIECE}.wav'):
    y1, _ = dj.librosa.load(f'output/generation_{PIECE}.wav', sr=dj.SR)
    print('v1 (07, grille mécanique) :'); display(Audio(y1, rate=dj.SR))
print('v2 (14, groove) :'); display(Audio(out, rate=dj.SR))
mix, _ = dj.build_mix(PIECE, 1)
print('original (référence) :'); display(Audio(mix[int(60*dj.SR):int(74*dj.SR)], rate=dj.SR))

## Bilan & horizon
- **v2 = groove** : ordre 2 + repli (phrases plus tenues), timing swingué, accelerando → ça respire beaucoup plus que la v1.
- Le générateur « parle djembé » de façon crédible dans le style de chaque pièce.
- **Prochain palier** (vrai LLM du djembé) : un petit **transformer** entraîné sur les partitions (06) capterait la structure longue (phrases de solo, chauffés, signaux de transition) que le Markov ne voit pas. Avec 3 rythmes × ~15 prises, données limitées → data augmentation ou pré-entraînement à envisager.